In [1]:
import glob
import numpy as np
import time
from PIL import Image
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchmetrics.classification import MulticlassConfusionMatrix
from torch.utils.data import Dataset, DataLoader

In [2]:
CLASSES = [ 
    'Car', 'Van', 'Truck', 'Pedestrian', 'Person_sitting', 'Cyclist', 'Tram', 'Misc'
]

RESIZE_TO = 640 

ANCHORS = [
    [(0.28, 0.22), (0.38, 0.48), (0.9, 0.78)],
    [(0.07, 0.15), (0.15, 0.11), (0.14, 0.29)],
]

S = [RESIZE_TO // 32, RESIZE_TO // 16]

NUM_CLASSES = len(CLASSES) 
NUM_WORKERS = os.cpu_count() // 2
BATCH_SIZE = 16
RESIZE_TO = 640 
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4 
PIN_MEMORY = True
CHECKPOINT_FILE = "./checkpoints/best.pth.tar"

test_images_path = "./dataset/train_example/images"

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [4]:

config = [
    (32, 3, 1),
    (64, 3, 2),
    ["B", 4],
    (128, 3, 2),
    ["B", 6],
    (256, 3, 2),
    ["B", 8],
    (512, 3, 2),
    ["B", 8],
    (1024, 3, 2),
    ["B", 6],
    (512, 1, 1),
    (1024, 3, 1),
    "S",
    (256, 1, 1),
    "U",
    (256, 1, 1),
    (512, 3, 1),
    "S",
]

class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, bn_act=True, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=not bn_act, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
        self.leaky = nn.LeakyReLU(0.1)
        self.use_bn_act = bn_act

    def forward(self, x):
        if self.use_bn_act:
            return self.leaky(self.bn(self.conv(x)))
        else:
            return self.conv(x)


class ResidualBlock(nn.Module):
    def __init__(self, channels, use_residual=True, num_repeats=1):
        super().__init__()
        self.layers = nn.ModuleList()
        for repeat in range(num_repeats):
            self.layers += [
                nn.Sequential(
                    CNNBlock(channels, channels // 2, kernel_size=1),
                    CNNBlock(channels // 2, channels, kernel_size=3, padding=1),
                )
            ]

        self.use_residual = use_residual
        self.num_repeats = num_repeats

    def forward(self, x):
        for layer in self.layers:
            if self.use_residual:
                x = x + layer(x)
            else:
                x = layer(x)

        return x


class ScalePrediction(nn.Module):
    def __init__(self, num_classes, in_channels=3):
        super().__init__()
        self.pred = nn.Sequential(
            CNNBlock(in_channels, 2 * in_channels, kernel_size=3, padding=1),
            CNNBlock(
                2 * in_channels, 3 * (num_classes + 5), bn_act=False, kernel_size=1
            ),
        )
        
        self.num_classes = num_classes

    def forward(self, x):
        return (
            self.pred(x)
            .reshape(x.shape[0], 3, self.num_classes + 5, x.shape[2], x.shape[3])
            .permute(0, 1, 3, 4, 2)
        )

class SiStNet(nn.Module):
    def __init__(self, num_classes, in_channels=3):
        super().__init__()
        self.num_classes = num_classes
        self.in_channels = in_channels
        self.layers = self._create_conv_layers()

    def forward(self, x):
        outputs = []
        route_connections = []
        for layer in self.layers:
            if isinstance(layer, ScalePrediction):
                outputs.append(layer(x))
                continue

            x = layer(x)

            if isinstance(layer, ResidualBlock) and layer.num_repeats == 8:
                route_connections.append(x)

            elif isinstance(layer, nn.Upsample):
                x = torch.cat([x, route_connections[-1]], dim=1)
                route_connections.pop()

        return outputs

    def _create_conv_layers(self):
        layers = nn.ModuleList()
        in_channels = self.in_channels
        
        for module in config:
            if isinstance(module, tuple):
                out_channels, kernel_size, stride = module
                layers.append(
                    CNNBlock(
                        in_channels,
                        out_channels,
                        kernel_size=kernel_size,
                        stride=stride,
                        padding=1 if kernel_size == 3 else 0,
                    )
                )
                in_channels = out_channels

            elif isinstance(module, list):
                num_repeats = module[1]
                layers.append(ResidualBlock(in_channels, num_repeats=num_repeats))

            elif isinstance(module, str):
                if module == "S":
                    layers += [
                        ResidualBlock(in_channels, use_residual=False, num_repeats=1),
                        CNNBlock(in_channels, in_channels // 2, kernel_size=1),
                        ScalePrediction(in_channels=in_channels // 2, num_classes=self.num_classes),
                    ]
                    in_channels = in_channels // 2

                elif module == "U":
                    layers.append(nn.Upsample(scale_factor=2))
                    in_channels = in_channels * 3

        return layers

if __name__ == "__main__": 
    model = SiStNet(num_classes=NUM_CLASSES)


In [5]:

test_transforms = A.Compose(
    [
        A.LongestMaxSize(max_size=RESIZE_TO),
        A.PadIfNeeded(
            min_height=RESIZE_TO, min_width=RESIZE_TO, border_mode=cv2.BORDER_CONSTANT
        ),
        A.Normalize(mean=[0, 0, 0], std=[1, 1, 1], max_pixel_value=255,),
        ToTensorV2(),
    ],
)

In [6]:
# ---------------------------
# 1. DEVICE
# ---------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------------------
# 2. MODELİ TANIMLA (SENİN MODEL)
# ---------------------------
# ⚠️ BU SATIR SİSTEMİNDE OLMALI
model = SiStNet(num_classes=NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ---------------------------
# 3. CHECKPOINT LOAD
# ---------------------------
def load_model(checkpoint_path):
    print("Loading model...")
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])

    model.eval()
    print("Model loaded successfully!")

load_model(CHECKPOINT_FILE)

# ---------------------------
# 4. INFERENCE DATASET (LABELSIZ)
# ---------------------------
class InferenceDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.images = glob.glob(f"{img_dir}/*")
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = np.array(Image.open(img_path).convert("RGB"))

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, img_path

# ---------------------------
# 5. INFERENCE + FPS + LATENCY
# ---------------------------
def run_benchmark(model, loader, device):
    model.eval()

    total_time = 0.0
    total_images = 0
    fps_list = []

    with torch.no_grad():
        for batch_idx, (images, _) in enumerate(loader):

            images = images.to(device)
            batch_size = images.shape[0]

            if device == "cuda":
                torch.cuda.synchronize()

            start = time.time()
            _ = model(images)

            if device == "cuda":
                torch.cuda.synchronize()

            end = time.time()

            batch_time = end - start

            total_time += batch_time
            total_images += batch_size

            fps = batch_size / batch_time
            fps_list.append(fps)

            print(f"Batch {batch_idx} | FPS: {fps:.2f} | Latency: {(batch_time/batch_size)*1000:.2f} ms/img")

    avg_fps = total_images / total_time
    avg_latency = (total_time / total_images) * 1000

    print("\n====================")
    print("FINAL RESULTS")
    print("====================")
    print(f"Avg FPS: {avg_fps:.2f}")
    print(f"Avg Latency: {avg_latency:.2f} ms/image")

    return avg_fps, avg_latency

# ---------------------------
# 6. DATASET + DATALOADER
# ---------------------------
test_dataset = InferenceDataset(
    img_dir=test_images_path,
    transform=test_transforms
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# ---------------------------
# 7. ANCHORS
# ---------------------------
scaled_anchors = (
    torch.tensor(ANCHORS)
    * torch.tensor(S).unsqueeze(1).unsqueeze(1).repeat(1, 3, 2)
).to(DEVICE)

# ---------------------------
# 8. RUN BENCHMARK
# ---------------------------
#avg_fps, avg_latency = run_benchmark(
#    model=model,
#    loader=test_loader,
#    anchors=scaled_anchors,
#    S=S
#)
avg_fps, avg_latency = run_benchmark(
    model=model,
    loader=test_loader,
    device=DEVICE
)

Loading model...
Model loaded successfully!
Batch 0 | FPS: 18.92 | Latency: 52.86 ms/img
Batch 1 | FPS: 26.41 | Latency: 37.87 ms/img
Batch 2 | FPS: 26.32 | Latency: 38.00 ms/img
Batch 3 | FPS: 26.29 | Latency: 38.04 ms/img
Batch 4 | FPS: 26.31 | Latency: 38.00 ms/img
Batch 5 | FPS: 26.31 | Latency: 38.00 ms/img
Batch 6 | FPS: 26.28 | Latency: 38.06 ms/img
Batch 7 | FPS: 26.33 | Latency: 37.98 ms/img
Batch 8 | FPS: 26.24 | Latency: 38.11 ms/img
Batch 9 | FPS: 26.27 | Latency: 38.06 ms/img
Batch 10 | FPS: 26.30 | Latency: 38.02 ms/img
Batch 11 | FPS: 26.23 | Latency: 38.12 ms/img
Batch 12 | FPS: 26.21 | Latency: 38.15 ms/img
Batch 13 | FPS: 26.25 | Latency: 38.09 ms/img
Batch 14 | FPS: 26.28 | Latency: 38.05 ms/img
Batch 15 | FPS: 26.23 | Latency: 38.12 ms/img
Batch 16 | FPS: 26.14 | Latency: 38.26 ms/img
Batch 17 | FPS: 26.23 | Latency: 38.13 ms/img
Batch 18 | FPS: 26.26 | Latency: 38.08 ms/img
Batch 19 | FPS: 26.29 | Latency: 38.03 ms/img
Batch 20 | FPS: 26.23 | Latency: 38.13 ms/img
